# DAAN 545 — Dimensionality reduction of delay profiles

**Goal:** Compress the same scaled delay-profile features used in clustering, with **PCA** (linear) and **t-SNE** (nonlinear), then check whether clustering in a reduced PCA space changes the groups.

**Input:** `data/processed/airline_delay_cause_with_airports.csv`. Same row filter and features as `notebooks/03_clustering.ipynb`.

**Approach:**
1. Reuse the clustering feature build (delay rates + cause shares; drop months below the 10th percentile of `arr_flights`). Standardize.
2. Fit **PCA**. Report variance by component and loadings.
3. Fit **t-SNE** in 2D (perplexity 30). Compare local-structure scores (trustworthiness) with 2D PCA.
4. Re-run **K-Means** in the original scaled space and in the PCs that hold at least 80% of variance. Compare silhouette and Adjusted Rand index.

Figures: `reports/figures/23_pca_variance.png` through `29_pca_loadings.png`.
Write-up language: `docs/notebook_notes.md`.


## 0. Setup

Helpers live in `src/dimred.py`. Feature construction is imported from `src/clustering.py` so both notebooks stay on the same matrix. Outlier notebooks are 04–05; this notebook does not replace them.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dimred import dimred_export_table, run_dimred, save_dimred_figures

DATA_PROCESSED = ROOT / "data" / "processed"
FIG_DIR = ROOT / "reports" / "figures"
print(f"Project root: {ROOT}")


## 1. Fit PCA and t-SNE

PCA is the linear baseline: variance explained and loadings. t-SNE is a 2D map meant to keep nearby months nearby. We do **not** cluster in t-SNE coordinates (distances there are not a global metric).


In [ ]:
panel = pd.read_csv(DATA_PROCESSED / "airline_delay_cause_with_airports.csv")
result = run_dimred(panel)

print("Rows clustered:", result["meta"]["n_rows_clustered"])
print("Features:", result["features"])
print("PCA variance by component:", result["pca_ratios"])
print("Cumulative:", result["pca_cumulative"])
print(f"PC1–PC2: {result['pca_variance_first_two']:.1%} (2D clustering rule ≥70%: {result['pca_cluster_in_2d']})")
print(f"PCs kept for 80% variance: {result['n_pcs_for_80pct']} ({result['variance_kept_for_clustering']:.1%})")
print(f"t-SNE perplexity: {result['tsne_perplexity']}")
print(f"Trustworthiness (5 neighbors): PCA-2D={result['trustworthiness_pca2']:.3f}, t-SNE={result['trustworthiness_tsne']:.3f}")
print("\nLoadings (first three PCs):")
display(result["loadings"].iloc[:, :3])


## 2. Clustering in original space vs PCA space

Same k as the silhouette rule in notebook 03. Labels from the full scaled matrix are compared with labels from the 80% PCA truncation.


In [ ]:
print("Chosen k:", result["k"])
print(f"Silhouette, K-Means on original features: {result['silhouette_original']:.3f}")
print(f"Silhouette, K-Means in PCA space (scored in PCA): {result['silhouette_pca_in_pca_space']:.3f}")
print(f"Those PCA labels scored back in original space: {result['silhouette_pca_labels_in_original']:.3f}")
print(f"Adjusted Rand (original K-Means vs PCA K-Means): {result['ari_original_vs_pca_kmeans']:.3f}")
print("\nSweep (PCA labels vs original labels at each k):")
display(
    result["orig_sweep"].merge(result["pca_k_sweep"], on="k").round(3)
)


## 3. Figures and exports

Scree plot, before/after 2D, 3D PCA, PCA vs t-SNE, clustering comparison, loadings (`23_` through `29_`).


In [ ]:
paths = save_dimred_figures(result, FIG_DIR)
for path in paths:
    print(path)

dimred_export_table(result).to_csv(DATA_PROCESSED / "dimred_coordinates.csv", index=False)
result["loadings"].to_csv(DATA_PROCESSED / "pca_loadings.csv")
result["orig_sweep"].merge(result["pca_k_sweep"], on="k").to_csv(
    DATA_PROCESSED / "dimred_cluster_compare.csv", index=False
)
print("Saved coordinates and PCA tables under data/processed/")
